<a href="https://colab.research.google.com/github/pepealania/agentic-rag/blob/main/PoC/LlamaIndexWorkflows2Nodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# LLAMAINDEX WORKFLOWS - Instalación
# ============================================================

!pip install -q "llama-index-core==0.14.23"

import sys
import llama_index.core

print("Python:", sys.version)
print("LlamaIndex Core:", llama_index.core.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.9/164.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
LlamaIndex Core: 0.14.23


In [4]:
# ============================================================
# LLAMAINDEX WORKFLOWS - PoC de dos nodos + DEBUGGING
# ============================================================

import time

from llama_index.core.workflow import (
    Workflow,
    StartEvent,
    StopEvent,
    Event,
    Context,
    step,
)


# ============================================================
# 1. EVENTO INTERMEDIO
# ============================================================

class EvidenciasEvent(Event):
    consulta: str
    evidencias: list[str]


# ============================================================
# 2. WORKFLOW
# ============================================================

class DosNodosWorkflow(Workflow):

    # --------------------------------------------------------
    # Nodo 1: recuperar evidencias
    # --------------------------------------------------------

    @step
    async def recuperar(
        self,
        ctx: Context,
        ev: StartEvent
    ) -> EvidenciasEvent:

        inicio = time.perf_counter()

        print("\n" + "=" * 60)
        print(">>> NODO 1: RECUPERAR")
        print("=" * 60)

        # ----------------------------------------------------
        # Entrada
        # ----------------------------------------------------

        consulta = ev.get("consulta")

        print(f"[DEBUG] Consulta recibida:")
        print(f"        {consulta}")

        # ----------------------------------------------------
        # Simulación de recuperación
        # ----------------------------------------------------

        evidencias = [
            "Documento A: LlamaIndex proporciona componentes para aplicaciones RAG.",
            "Documento B: Workflows permiten orquestar pasos mediante eventos."
        ]

        print(f"\n[DEBUG] Evidencias recuperadas: {len(evidencias)}")

        for i, evidencia in enumerate(evidencias, 1):
            print(f"        [{i}] {evidencia}")

        # ----------------------------------------------------
        # Guardar estado en Context
        # ----------------------------------------------------

        await ctx.store.set(
            "consulta",
            consulta
        )

        await ctx.store.set(
            "evidencias",
            evidencias
        )

        # Historial de ejecución
        historial = await ctx.store.get(
            "historial",
            default=[]
        )

        historial.append({
            "nodo": "recuperar",
            "entrada": consulta,
            "evidencias_generadas": len(evidencias),
        })

        await ctx.store.set(
            "historial",
            historial
        )

        # ----------------------------------------------------
        # Tiempo de ejecución
        # ----------------------------------------------------

        tiempo = time.perf_counter() - inicio

        print(f"\n[DEBUG] Tiempo nodo recuperar: {tiempo:.6f} s")

        return EvidenciasEvent(
            consulta=consulta,
            evidencias=evidencias
        )


    # --------------------------------------------------------
    # Nodo 2: analizar
    # --------------------------------------------------------

    @step
    async def analizar(
        self,
        ctx: Context,
        ev: EvidenciasEvent
    ) -> StopEvent:

        inicio = time.perf_counter()

        print("\n" + "=" * 60)
        print(">>> NODO 2: ANALIZAR")
        print("=" * 60)

        # ----------------------------------------------------
        # Inspección de entrada
        # ----------------------------------------------------

        print("[DEBUG] Consulta:")
        print(f"        {ev.consulta}")

        print("\n[DEBUG] Evidencias recibidas:")
        for i, evidencia in enumerate(ev.evidencias, 1):
            print(f"        [{i}] {evidencia}")

        # ----------------------------------------------------
        # Análisis
        # ----------------------------------------------------

        resultado = (
            f"Consulta: {ev.consulta}\n"
            f"Evidencias encontradas: {len(ev.evidencias)}\n"
            f"Conclusión: las evidencias son suficientes para continuar."
        )

        print("\n[DEBUG] Resultado generado:")
        print(resultado)

        # ----------------------------------------------------
        # Guardar resultado en Context
        # ----------------------------------------------------

        await ctx.store.set(
            "resultado",
            resultado
        )

        historial = await ctx.store.get(
            "historial",
            default=[]
        )

        historial.append({
            "nodo": "analizar",
            "evidencias_recibidas": len(ev.evidencias),
            "resultado_generado": resultado,
        })

        await ctx.store.set(
            "historial",
            historial
        )

        # ----------------------------------------------------
        # Tiempo
        # ----------------------------------------------------

        tiempo = time.perf_counter() - inicio

        print(f"\n[DEBUG] Tiempo nodo analizar: {tiempo:.6f} s")

        return StopEvent(
            result={
                "consulta": ev.consulta,
                "evidencias": ev.evidencias,
                "resultado": resultado,
            }
        )


# ============================================================
# 3. EJECUCIÓN
# ============================================================

workflow = DosNodosWorkflow(timeout=30)

# Crear contexto para conservar estado
ctx = Context(workflow)


# ============================================================
# 4. EJECUTAR WORKFLOW
# ============================================================

result = await workflow.run(
    consulta="¿Qué framework facilita la construcción de aplicaciones RAG?",
    ctx=ctx
)


# ============================================================
# 5. RESULTADO FINAL
# ============================================================

print("\n")
print("=" * 60)
print("RESULTADO FINAL")
print("=" * 60)

print(result["resultado"])


# ============================================================
# 6. INSPECCIONAR ESTADO
# ============================================================

print("\n")
print("=" * 60)
print("ESTADO DEL WORKFLOW")
print("=" * 60)

consulta = await ctx.store.get("consulta")
evidencias = await ctx.store.get("evidencias")
resultado = await ctx.store.get("resultado")

print("\nConsulta:")
print(consulta)

print("\nEvidencias:")
for evidencia in evidencias:
    print("-", evidencia)

print("\nResultado:")
print(resultado)


# ============================================================
# 7. HISTORIAL DE EJECUCIÓN
# ============================================================

print("\n")
print("=" * 60)
print("HISTORIAL DE EJECUCIÓN")
print("=" * 60)

historial = await ctx.store.get(
    "historial",
    default=[]
)

for i, paso in enumerate(historial, 1):

    print(f"\n--- Paso {i} ---")

    for clave, valor in paso.items():
        print(f"{clave}: {valor}")


>>> NODO 1: RECUPERAR
[DEBUG] Consulta recibida:
        ¿Qué framework facilita la construcción de aplicaciones RAG?

[DEBUG] Evidencias recuperadas: 2
        [1] Documento A: LlamaIndex proporciona componentes para aplicaciones RAG.
        [2] Documento B: Workflows permiten orquestar pasos mediante eventos.

[DEBUG] Tiempo nodo recuperar: 0.000541 s

>>> NODO 2: ANALIZAR
[DEBUG] Consulta:
        ¿Qué framework facilita la construcción de aplicaciones RAG?

[DEBUG] Evidencias recibidas:
        [1] Documento A: LlamaIndex proporciona componentes para aplicaciones RAG.
        [2] Documento B: Workflows permiten orquestar pasos mediante eventos.

[DEBUG] Resultado generado:
Consulta: ¿Qué framework facilita la construcción de aplicaciones RAG?
Evidencias encontradas: 2
Conclusión: las evidencias son suficientes para continuar.

[DEBUG] Tiempo nodo analizar: 0.000300 s


RESULTADO FINAL
Consulta: ¿Qué framework facilita la construcción de aplicaciones RAG?
Evidencias encontradas: 2